# 02 — Depth, nonlinearity, and overfitting (MLP)

Two lessons. First, **why nonlinearity matters**: on data that isn't linearly separable, a linear model fails and an MLP (Linear → nonlinearity → Linear) succeeds. Second, the central practical tension of training — **overfitting** — and the regularizers that control it.

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Data — concentric circles (not linearly separable)

Inner disk vs. outer ring: no straight line separates them, so a linear model is stuck near chance.

In [ ]:
def make_circles(n, noise=0.12):
    lab = np.random.randint(0, 2, n)
    r = np.where(lab == 0, np.random.rand(n)*0.7, 1.3 + np.random.rand(n)*0.7)
    th = np.random.rand(n) * 2*np.pi
    X = np.stack([r*np.cos(th), r*np.sin(th)], 1).astype(np.float32) + np.random.randn(n, 2).astype(np.float32)*noise
    return torch.tensor(X), torch.tensor(lab)

Xtr, ytr = make_circles(1500); Xva, yva = make_circles(500)
Xtr, ytr, Xva, yva = (t.to(device) for t in (Xtr, ytr, Xva, yva))
plt.figure(figsize=(4, 4))
plt.scatter(Xtr[:, 0].cpu(), Xtr[:, 1].cpu(), c=ytr.cpu(), s=8, cmap="coolwarm")
plt.title("concentric circles"); plt.show()

## A reusable train/eval helper

Same loop as notebook 01, packaged so we can train several models quickly. It records per-epoch train/val loss and returns final accuracies.

In [ ]:
def fit(model, Xtr, ytr, Xva, yva, epochs, lr=1e-2, wd=0.0, bs=64):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    hist = {"tr": [], "va": []}
    for _ in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for k in range(0, len(Xtr), bs):
            idx = perm[k:k+bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            hist["tr"].append(F.cross_entropy(model(Xtr), ytr).item())
            hist["va"].append(F.cross_entropy(model(Xva), yva).item())
    with torch.no_grad():
        ta = (model(Xtr).argmax(1) == ytr).float().mean().item()
        va = (model(Xva).argmax(1) == yva).float().mean().item()
    return model, hist, ta, va

## Linear vs. MLP on the circles

In [ ]:
_, _, lin_ta, lin_va = fit(nn.Linear(2, 2), Xtr, ytr, Xva, yva, epochs=60)
print(f"linear:  train {lin_ta:.3f}  val {lin_va:.3f}   <- stuck near chance")

mlp = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
_, _, mlp_ta, mlp_va = fit(mlp, Xtr, ytr, Xva, yva, epochs=60)
print(f"MLP:     train {mlp_ta:.3f}  val {mlp_va:.3f}   <- nonlinearity bends the boundary")

The `ReLU` between linear layers is what lets the MLP carve a curved boundary — stack of linears alone collapses to one linear map. That's the whole point of a nonlinearity.

## Overfitting — the central tension

Give a **big** model a **small, noisy** training set and it will *memorize* it (train accuracy → ~1.0) while generalizing worse. The tell-tale sign is the **validation loss turning back up** even as train loss keeps falling.

In [ ]:
# small noisy training set + 15% label noise; large held-out val
lab_noise = 0.15
Xs, ys = make_circles(120, noise=0.25)
flip = torch.rand(120) < lab_noise; ys = ys.clone(); ys[flip] = 1 - ys[flip]
Xsv, ysv = make_circles(2000, noise=0.25)
Xs, ys, Xsv, ysv = (t.to(device) for t in (Xs, ys, Xsv, ysv))

def big(): return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

_, h_none, ta0, va0 = fit(big(), Xs, ys, Xsv, ysv, epochs=400, lr=5e-3, wd=0.0)
_, h_wd,   ta1, va1 = fit(big(), Xs, ys, Xsv, ysv, epochs=400, lr=5e-3, wd=5e-2)   # weight decay
print(f"no reg:       train_acc {ta0:.3f}  val_acc {va0:.3f}  final val_loss {h_none['va'][-1]:.3f}")
print(f"weight decay: train_acc {ta1:.3f}  val_acc {va1:.3f}  final val_loss {h_wd['va'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(5.5, 3.5))
plt.plot(h_none["tr"], "C0--", label="train (no reg)")
plt.plot(h_none["va"], "C0-",  label="val (no reg)")
plt.plot(h_wd["va"],   "C1-",  label="val (weight decay)")
plt.title("overfitting: val loss turns UP; weight decay tames it")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

See it: **train loss keeps dropping** (the model memorizes, including the noise) while the **no-reg val loss bottoms out and rises** — classic overfitting. **Weight decay** (the `wd` in AdamW, Part 2.4) keeps the val loss lower for longer.

## The regularization toolkit (all "without complicated tricks")

- **Weight decay** — shrink weights each step; discourages memorizing (used above).
- **Dropout** (`nn.Dropout(p)`) — randomly zero activations during training; forces redundancy. (Active in `train()`, off in `eval()` — a reason those modes exist.)
- **Early stopping** — stop at the val-loss minimum (the dip before the upturn above).
- **More data** — the most reliable fix of all.

## Takeaways

- Nonlinearity (`ReLU`) is what makes depth expressive — without it, layers collapse to one linear map.
- Overfitting = great train, worse val; **watch val loss**, not just train.
- Weight decay / dropout / early stopping / more data all pull val performance back up.

Next: from 2-D points to **sequences of tokens** — embeddings and pooling (notebook 03).